# Тема 3. Классификация: деревья решений и метод ближайших соседей

## Содержание

1. [Введение: задачи машинного обучения](#1.-Введение:-задачи-машинного-обучения)
2. [Дерево решений](#2.-Дерево-решений)
3. [Метод ближайших соседей (k-NN)](#3.-Метод-ближайших-соседей)
4. [Выбор параметров модели и кросс-валидация](#4.-Выбор-параметров-модели-и-кросс-валидация)
5. [Примеры на реальных данных и сложные случаи](#5.-Примеры-на-реальных-данных-и-сложные-случаи)
6. [Плюсы и минусы методов](#6.-Плюсы-и-минусы-методов)
7. [Полезные ресурсы](#7.-Полезные-ресурсы)

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
sns.set()
from matplotlib import pyplot as plt

%config InlineBackend.figure_format = 'svg'

---
## 1. Введение: задачи машинного обучения

Томас Митчелл в книге «Machine Learning» (1997) дал такое классическое определение:

> Говорят, что компьютерная программа *обучается* на опыте **E** относительно класса задач **T** и меры качества **P**, если её производительность при решении задач из **T**, измеряемая по **P**, улучшается с опытом **E**.

Что скрывается за этими буквами:

- **T (Task)** — задача: *классификация*, *регрессия*, *кластеризация*, *обнаружение аномалий* и многие другие.
- **E (Experience)** — данные, на которых учится модель.
- **P (Performance)** — метрика качества: доля правильных ответов (accuracy), средняя ошибка (MSE) и т.д.

Сегодня мы работаем с **задачей классификации** — предсказываем, к какому классу принадлежит объект.

#### Пример

Банк хочет предсказывать дефолт по кредиту. Данные о клиенте (возраст, зарплата, история платежей) — это признаки **X**. Факт дефолта (да/нет) — целевая переменная **y**. Задача бинарной классификации.

---
## 2. Дерево решений

Деревья решений используются в повседневной жизни, не только в ML. Любая блок-схема — это, по сути, дерево решений.

Вот пример простого дерева для кредитного скоринга:

```
                    Возраст > 40?
                   /             \
                Да                Нет
               /                   \
        Есть ли дом?           Доход > 5000?
        /         \             /          \
      Да           Нет        Да            Нет
    Выдать       Отказать   Выдать        Отказать
```

Преимущество очевидно: такую модель легко объяснить клиенту. «Вам отказано, потому что вы моложе 40 лет и ваш доход ниже 5000». Именно поэтому деревья решений исторически популярны в задачах, где важна интерпретируемость (медицина, финансы, юриспруденция).

### 2.1 Как строится дерево решений

Вспомните игру «20 вопросов»: один загадывает знаменитость, другой угадывает, задавая вопросы «да/нет». Какой вопрос задать первым?

Очевидно — тот, который сильнее сократит пространство вариантов. Вопрос «Это Анджелина Джоли?» при ответе «нет» оставляет почти всех знаменитостей. Вопрос «Это женщина?» сразу делит список пополам. Признак «пол» **разделяет** данные намного лучше.

Это интуиция за понятием **прироста информации** (information gain), основанного на **энтропии**.

### 2.2 Энтропия

Энтропия Шеннона для системы с $N$ возможными состояниями:

$$S = -\sum_{i=1}^{N} p_i \log_2 p_i$$

где $p_i$ — вероятность $i$-го состояния.

**Физический смысл:** энтропия — мера хаоса или неопределённости. Чем выше энтропия, тем меньше мы знаем о том, что произойдёт.

- Если в наборе все объекты одного класса → энтропия = 0 (полная определённость)
- Если классов поровну → энтропия максимальна (= 1 для бинарной задачи)

### 2.3 Игрушечный пример: шарики

Представьте: 9 синих и 11 жёлтых шаров расположены по оси $x$. Нужно предсказать цвет шара по его позиции.

**Начальная энтропия** (до любых разбиений):
$$S_0 = -\frac{9}{20}\log_2\frac{9}{20} - \frac{11}{20}\log_2\frac{11}{20} \approx 1$$

Попробуем разбить по условию $x \leq 12$:

- **Левая группа** (13 шаров: 8 синих, 5 жёлтых): $S_1 \approx 0.96$
- **Правая группа** (7 шаров: 1 синий, 6 жёлтых): $S_2 \approx 0.60$

**Прирост информации** (Information Gain):

$$IG = S_0 - \frac{N_1}{N} S_1 - \frac{N_2}{N} S_2 = 1 - \frac{13}{20} \cdot 0.96 - \frac{7}{20} \cdot 0.60 \approx 0.16$$

Разбиение уменьшило «хаос» в системе. Алгоритм будет продолжать делить группы, пока энтропия каждого листа не станет равной 0 (все объекты одного класса) или не достигнет заданного порога.

### 2.4 Алгоритм построения дерева

В основе популярных алгоритмов (ID3, C4.5, CART) лежит принцип **жадной максимизации прироста информации**:

```
def build(data):
    создать узел t
    если выполняется критерий остановки:
        назначить узлу предсказание (класс большинства)
    иначе:
        найти наилучшее разбиение data = data_left + data_right
        t.left  = build(data_left)
        t.right = build(data_right)
    вернуть t
```

На каждом шаге перебираются все признаки и все возможные пороги — выбирается тот, что даёт наибольший IG. Процесс рекурсивный.

### 2.5 Критерии качества разбиения

Энтропия — не единственный критерий. На практике используют:

| Критерий | Формула |
|---|---|
| **Энтропия** | $S = -\sum_k p_k \log_2 p_k$ |
| **Критерий Джини** | $G = 1 - \sum_k p_k^2$ |
| **Ошибка классификации** | $E = 1 - \max_k p_k$ |

Для бинарной классификации ($p_+$ — доля объектов класса «+»):

$$S = -p_+ \log_2 p_+ - (1-p_+) \log_2 (1-p_+)$$
$$G = 2 p_+ (1 - p_+)$$

Нарисуем эти функции:

In [ ]:
plt.figure(figsize=(7, 4))
xx = np.linspace(0.01, 0.99, 100)
plt.plot(xx, [2 * x * (1 - x) for x in xx], label="Джини")
plt.plot(xx, [4 * x * (1 - x) for x in xx], label="2 × Джини")
plt.plot(xx, [-x * np.log2(x) - (1 - x) * np.log2(1 - x) for x in xx], label="Энтропия")
plt.plot(xx, [1 - max(x, 1 - x) for x in xx], label="Ошибка класс-и")
plt.xlabel("$p_+$")
plt.ylabel("Критерий")
plt.title("Критерии качества разбиения (бинарная классификация)")
plt.legend();


Видно, что энтропия и критерий Джини (умноженный на 2) ведут себя почти одинаково — на практике результаты с ними совпадают. Ошибка классификации хуже улавливает различия при малом $p_+$, поэтому её используют реже.

### 2.6 Пример: классификация синтетических данных

Сгенерируем два класса — два нормальных распределения с разными центрами — и посмотрим, какую границу строит дерево решений.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

np.random.seed(17)
train_data = np.random.normal(size=(100, 2))
train_labels = np.zeros(100)

train_data = np.r_[train_data, np.random.normal(size=(100, 2), loc=2)]
train_labels = np.r_[train_labels, np.ones(100)]

plt.figure(figsize=(8, 6))
plt.scatter(train_data[:, 0], train_data[:, 1],
            c=train_labels, s=80, cmap="autumn", edgecolors="black", linewidth=1)
plt.title("Два класса — два нормальных распределения")
plt.xlabel("x1"); plt.ylabel("x2");

In [ ]:
def get_grid(data, step=0.02):
    x_min, x_max = data[:, 0].min() - 1, data[:, 0].max() + 1
    y_min, y_max = data[:, 1].min() - 1, data[:, 1].max() + 1
    return np.meshgrid(np.arange(x_min, x_max, step),
                       np.arange(y_min, y_max, step))

clf_tree = DecisionTreeClassifier(criterion="entropy", max_depth=3, random_state=17)
clf_tree.fit(train_data, train_labels)

xx, yy = get_grid(train_data)
predicted = clf_tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.pcolormesh(xx, yy, predicted, cmap="autumn", alpha=0.3)
plt.scatter(train_data[:, 0], train_data[:, 1],
            c=train_labels, s=80, cmap="autumn", edgecolors="black", linewidth=1)
plt.title("Граница решений дерева (max_depth=3)");

Дерево глубины 3 разбивает пространство на $2^3 = 8$ прямоугольных областей. Внутри каждой области предсказывается класс большинства. Такая ступенчатая граница — характерная черта деревьев решений: она всегда параллельна осям координат.

### 2.7 Работа с числовыми признаками

Если признак числовой и имеет много уникальных значений (например, возраст), как дерево выбирает порог сравнения?

**Ключевое правило:** дерево ищет пороги только там, где меняется значение целевой переменной при сортировке объектов по данному признаку.

Пример:

In [ ]:
data = pd.DataFrame({
    "Возраст":       [17, 64, 18, 20, 38, 49, 55, 25, 29, 31, 33],
    "Дефолт по кредиту": [1,  0,  1,  0,  1,  0,  0,  1,  1,  0,  1],
})
data.sort_values("Возраст")

Посмотрим на последовательность дефолтов при сортировке по возрасту:

```
Возраст:   17  18  20  25  29  31  33  38  49  55  64
Дефолт:     1   1   0   1   1   0   1   1   0   0   0
                     ↑       ↑    ↑   ↑    ↑
                  смена    смена смена смена смена
```

Пороги для проверки: средние значения между точками смены → 19, 22.5, 30, 32, 43.5 лет.

**Вывод:** при большом числе числовых признаков, каждый из которых может иметь много уникальных значений, дерево проверяет только $O(n)$ порогов (по числу объектов), а не все возможные числа.

In [ ]:
data2 = pd.DataFrame({
    "Возраст":   [17, 64, 18, 20, 38, 49, 55, 25, 29, 31, 33],
    "Зарплата":  [25, 80, 22, 36, 37, 59, 74, 70, 33,102, 88],
    "Дефолт":    [ 1,  0,  1,  0,  1,  0,  0,  1,  1,  0,  1],
})
data2.sort_values("Возраст")

При двух числовых признаках дерево будет на каждом шаге выбирать, по какому из них делать следующее разбиение — тому, что даёт больший прирост информации.

### 2.8 Переобучение и ключевые параметры

Теоретически можно строить дерево до тех пор, пока в каждом листе не окажется ровно один объект. Тогда на обучающей выборке будет 100% точность. Но на новых данных такая модель ошибётся — она выучила шум.

Это называется **переобучением** (overfitting).

Представьте: дерево обнаружило, что все 4 клиента банка, пришедшие в зелёных брюках, не вернули кредит. Даже если это правда в обучающей выборке, такое правило бесполезно на новых клиентах.

**Способы борьбы с переобучением в деревьях:**
- Ограничение глубины (`max_depth`)
- Минимальное число объектов в листе (`min_samples_leaf`)  
- Минимальное число объектов для разбиения узла (`min_samples_split`)
- Отсечение (pruning) — строим полное дерево, потом обрезаем снизу вверх

**Основные параметры `DecisionTreeClassifier` в sklearn:**

| Параметр | Что контролирует |
|---|---|
| `max_depth` | Максимальная глубина дерева |
| `min_samples_leaf` | Минимальное число объектов в листе |
| `max_features` | Число признаков, рассматриваемых при каждом разбиении |
| `criterion` | Критерий качества: `"gini"` или `"entropy"` |

Правильные значения параметров подбираются через кросс-валидацию (см. раздел 4).

### 2.9 Дерево решений для регрессии

Дерево работает и для задач регрессии (предсказание числа). Идея та же, но критерий качества разбиения — **дисперсия** целевой переменной внутри каждого листа:

$$D = \frac{1}{\ell} \sum_{i=1}^{\ell} \left(y_i - \bar{y}\right)^2$$

Цель: найти разбиение, при котором значения $y$ в каждом листе максимально однородны (дисперсия минимальна). Предсказание в листе — среднее значение $\bar{y}$ по объектам этого листа.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

def f(x):
    x = x.ravel()
    return np.exp(-(x ** 2)) + 1.5 * np.exp(-((x - 2) ** 2))

def generate(n_samples, noise=0.1):
    X = np.sort(np.random.rand(n_samples) * 10 - 5)
    y = f(X) + np.random.normal(0, noise, n_samples)
    return X.reshape(-1, 1), y

np.random.seed(17)
X_train, y_train = generate(150)
X_test,  y_test  = generate(1000)

reg_tree = DecisionTreeRegressor(max_depth=5, random_state=17)
reg_tree.fit(X_train, y_train)
pred = reg_tree.predict(X_test)

plt.figure(figsize=(10, 5))
plt.plot(X_test, f(X_test), "b-", label="Истинная функция")
plt.scatter(X_train, y_train, c="b", s=15, alpha=0.6, label="Обучающие данные")
plt.plot(X_test, pred, "g-", lw=2, label="Предсказание дерева")
plt.xlim([-5, 5])
mse = np.mean((y_test - pred) ** 2)
plt.title(f"Регрессионное дерево (max_depth=5), MSE = {mse:.3f}")
plt.legend();

Дерево аппроксимирует функцию **кусочно-константной** функцией: в каждом листе предсказывается одно число. Чем глубже дерево, тем больше «ступенек» и точнее аппроксимация на обучающей выборке — но тем выше риск переобучения.

---
## 3. Метод ближайших соседей (k-NN)

Метод $k$ ближайших соседей (k-Nearest Neighbors) — один из самых интуитивных алгоритмов машинного обучения. Его основная идея: **ты похож на своих соседей**.

Формально: гипотеза компактности — если расстояние между объектами измерено хорошо, то похожие объекты, скорее всего, принадлежат одному классу.

**Алгоритм классификации:**
1. Для каждого тестового объекта вычислить расстояние до каждого объекта обучающей выборки
2. Выбрать $k$ ближайших соседей
3. Предсказать класс — тот, что встречается среди $k$ соседей чаще всего

**Для регрессии:** на шаге 3 вместо класса возвращается среднее (или медиана) значений целевой переменной среди $k$ соседей.

**Важная особенность:** k-NN — «ленивый» алгоритм. На этапе обучения он не строит никакой модели — просто запоминает все данные. Все вычисления происходят только в момент предсказания.

### 3.1 Ключевые параметры k-NN

**Число соседей $k$:**
- Маленькое $k$ → модель чувствительна к выбросам, переобучение
- Большое $k$ → слишком «усреднённые» предсказания, недообучение

**Метрика расстояния:**
- Евклидово: $d = \sqrt{\sum_i (x_i - z_i)^2}$
- Манхэттенское: $d = \sum_i |x_i - z_i|$
- Минковского (обобщение): $d = \left(\sum_i |x_i - z_i|^p\right)^{1/p}$

⚠️ **Важно:** большинство метрик расстояния чувствительны к масштабу признаков. Признак «зарплата» (тысячи) будет доминировать над «возрастом» (десятки). Поэтому **перед k-NN данные нужно масштабировать** (например, `StandardScaler`).

**Веса соседей:**
- `uniform` — все соседи равноправны
- `distance` — вес обратно пропорционален расстоянию (ближние важнее)

### 3.2 Параметры `KNeighborsClassifier` в sklearn

| Параметр | Что контролирует |
|---|---|
| `n_neighbors` | Число соседей $k$ |
| `weights` | Веса: `'uniform'` или `'distance'` |
| `metric` | Метрика расстояния |
| `algorithm` | Алгоритм поиска: `'brute'`, `'ball_tree'`, `'kd_tree'`, `'auto'` |

---
## 4. Выбор параметров модели и кросс-валидация

Для обоих алгоритмов нужно подбирать параметры: глубину дерева, число соседей и т.д. Как это делать честно?

### 4.1 Hold-out (отложенная выборка)

Самый простой подход: разделить данные на **обучающую** и **тестовую** части.

```
Все данные:  [======================================]
              [==== train (70-80%) ====][= test (20-30%) =]
```

Обучаем модель на train, оцениваем на test. Тест не участвует в обучении — это честная оценка.

**Проблема:** оценка зависит от того, как именно мы разделили данные. Если тест попался «удачным», мы переоцениваем качество модели.

### 4.2 Кросс-валидация (Cross-Validation)

Более надёжный подход — **$k$-fold cross-validation**:

```
Fold 1: [TEST][----train-----]
Fold 2: [----][TEST][--train-]
Fold 3: [----train--][TEST][-]
Fold 4: [--train----][----][TEST]
Fold 5: [----train-------][TEST]
```

1. Делим данные на $k$ равных частей (folds)
2. $k$ раз обучаем модель: каждый раз одна часть — тест, остальные — обучение
3. Итоговое качество = среднее по $k$ запускам

Типичные значения $k$: 5 или 10.

**Плюсы кросс-валидации:**
- Более надёжная оценка (меньше зависит от случайного разбиения)
- Каждый объект побывал в тесте ровно один раз

**Минусы:**
- В $k$ раз дольше обучение (можно параллелить, `n_jobs=-1`)

Кросс-валидация используется для **подбора гиперпараметров**: запускаем `GridSearchCV` — сетку по всем комбинациям параметров, оцениваем каждую через CV, берём лучшую.

---
## 5. Примеры на реальных данных и сложные случаи

### 5.1 Предсказание оттока клиентов телекома

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline

df = pd.read_csv("../../data/telecom_churn.csv")

# Кодируем бинарные признаки числами
df["International plan"] = pd.factorize(df["International plan"])[0]
df["Voice mail plan"]    = pd.factorize(df["Voice mail plan"])[0]
df["Churn"] = df["Churn"].astype("int")

states = df["State"]
y = df["Churn"]
df.drop(["State", "Churn"], axis=1, inplace=True)

df.head()

In [ ]:
# Разбиваем: 70% — обучение, 30% — отложенная выборка (holdout)
X_train, X_holdout, y_train, y_holdout = train_test_split(
    df.values, y, test_size=0.3, random_state=17
)

# Дерево с произвольными параметрами
tree = DecisionTreeClassifier(max_depth=5, random_state=17)

# k-NN — не забываем масштабировать!
scaler = StandardScaler()
X_train_scaled   = scaler.fit_transform(X_train)
X_holdout_scaled = scaler.transform(X_holdout)
knn = KNeighborsClassifier(n_neighbors=10)

tree.fit(X_train, y_train)
knn.fit(X_train_scaled, y_train)

In [ ]:
tree_pred = tree.predict(X_holdout)
knn_pred  = knn.predict(X_holdout_scaled)

print(f"Дерево решений (max_depth=5):  accuracy = {accuracy_score(y_holdout, tree_pred):.3f}")
print(f"k-NN (k=10):                   accuracy = {accuracy_score(y_holdout, knn_pred):.3f}")

Дерево с «угаданными» параметрами уже работает неплохо. Теперь подберём параметры через кросс-валидацию.

In [ ]:
# GridSearchCV перебирает все комбинации параметров и оценивает через 5-fold CV
tree_params = {
    "max_depth":    range(1, 11),
    "max_features": range(4, 19),
}

tree_grid = GridSearchCV(tree, tree_params, cv=5, n_jobs=-1, verbose=1)
tree_grid.fit(X_train, y_train)

In [ ]:
print("Лучшие параметры:", tree_grid.best_params_)
print(f"CV accuracy:      {tree_grid.best_score_:.4f}")
print(f"Holdout accuracy: {accuracy_score(y_holdout, tree_grid.predict(X_holdout)):.4f}")

Теперь то же самое для k-NN. Используем `Pipeline`, чтобы масштабирование и классификатор шли вместе — это гарантирует, что scaler будет обучаться только на train-части каждого fold.

In [ ]:
knn_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_jobs=-1))
])

knn_params = {"knn__n_neighbors": range(1, 20)}

knn_grid = GridSearchCV(knn_pipe, knn_params, cv=5, n_jobs=-1, verbose=1)
knn_grid.fit(X_train, y_train)

In [ ]:
print("Лучшие параметры:", knn_grid.best_params_)
print(f"CV accuracy:      {knn_grid.best_score_:.4f}")
print(f"Holdout accuracy: {accuracy_score(y_holdout, knn_grid.predict(X_holdout)):.4f}")

Дерево решений явно лучше k-NN на этом датасете. Для сравнения — быстро проверим случайный лес:

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=17)
cv_score = np.mean(cross_val_score(forest, X_train, y_train, cv=5))
forest.fit(X_train, y_train)
holdout_score = accuracy_score(y_holdout, forest.predict(X_holdout))

print(f"Random Forest — CV: {cv_score:.4f}, Holdout: {holdout_score:.4f}")

**Итоги по задаче оттока:**

| Модель | CV | Holdout |
|---|---|---|
| Дерево (подобр. параметры) | ~0.942 | ~0.946 |
| k-NN (подобр. параметры) | ~0.885 | ~0.890 |
| Случайный лес | ~0.951 | ~0.953 |

Дерево работает очень хорошо, случайный лес немного лучше. k-NN заметно хуже.

**Вывод:** всегда начинайте с простых моделей. Дерево решений и k-NN — хорошая отправная точка. Возможно, вам не понадобится ничего сложнее.

### 5.2 Сложный случай для дерева решений

Рассмотрим задачу, в которой классы линейно разделимы — достаточно провести одну прямую. Дерево должно справиться с этим легко. Посмотрим, что получится:

In [ ]:
def form_linearly_separable_data(n=500):
    data, target = [], []
    for _ in range(n):
        x1 = np.random.randint(0, 30)
        x2 = np.random.randint(0, 30)
        if abs(x1 - x2) > 0.5:
            data.append([x1, x2])
            target.append(np.sign(x1 - x2))
    return np.array(data), np.array(target)

np.random.seed(42)
X, y = form_linearly_separable_data()

plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="autumn", edgecolors="black", s=30)
plt.title("Линейно разделимые данные");

In [ ]:
tree = DecisionTreeClassifier(random_state=17).fit(X, y)

xx, yy = get_grid(X, step=0.5)
predicted = tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.pcolormesh(xx, yy, predicted, cmap="autumn", alpha=0.3)
plt.scatter(X[:, 0], X[:, 1], c=y, s=40, cmap="autumn", edgecolors="black", linewidth=1)
plt.title("Дерево решений усложняет простую задачу");

Граница решений — сложная ступенчатая конструкция, хотя истинная граница — просто прямая $x_1 = x_2$. Дерево справляется с задачей, но делает это избыточно сложно.

Почему? Потому что дерево **всегда строит границы параллельно осям координат**. Для диагональной прямой это требует множества горизонтальных и вертикальных «ступенек». Такая граница будет хорошо работать внутри обучающего квадрата $30 \times 30$, но плохо — за его пределами.

Это фундаментальное ограничение деревьев. Линейный классификатор (следующая тема) справился бы с этой задачей идеально.

### 5.3 Распознавание рукописных цифр (MNIST)

Теперь посмотрим на задачу, где k-NN, наоборот, работает неожиданно хорошо.

Датасет MNIST: изображения рукописных цифр 8×8 пикселей. Каждое изображение «разворачивается» в вектор длиной 64 — это и есть признаки.

In [ ]:
from sklearn.datasets import load_digits

data = load_digits()
X, y = data.data, data.target

# Посмотрим на несколько цифр
fig, axes = plt.subplots(1, 8, figsize=(16, 3))
for i in range(8):
    axes[i].imshow(X[i].reshape(8, 8), cmap="Greys")
    axes[i].set_title(f"Цифра: {y[i]}")
    axes[i].axis("off")

In [ ]:
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.3, random_state=17
)

# Случайные параметры — посмотрим на baseline
tree = DecisionTreeClassifier(max_depth=5, random_state=17)
knn_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=10))
])

tree.fit(X_train, y_train)
knn_pipe.fit(X_train, y_train)

tree_acc = accuracy_score(y_holdout, tree.predict(X_holdout))
knn_acc  = accuracy_score(y_holdout, knn_pipe.predict(X_holdout))
print(f"Дерево (max_depth=5): {tree_acc:.3f}")
print(f"k-NN (k=10):          {knn_acc:.3f}")

k-NN уже с «угаданными» параметрами намного лучше. Подберём параметры:

In [ ]:
tree_params = {
    "max_depth":    [5, 10, 20, 30, 40, 50, 64],
    "max_features": [5, 10, 20, 30, 50, 64],
}
tree_grid = GridSearchCV(tree, tree_params, cv=5, n_jobs=-1)
tree_grid.fit(X_train, y_train)
print("Дерево — лучшие параметры:", tree_grid.best_params_)
print(f"CV: {tree_grid.best_score_:.3f}, Holdout: {accuracy_score(y_holdout, tree_grid.predict(X_holdout)):.3f}")

In [ ]:
# k-NN с 1 соседом — часто работает очень хорошо для изображений
knn_1 = KNeighborsClassifier(n_neighbors=1)
cv_score_knn1 = np.mean(cross_val_score(knn_1, X_train, y_train, cv=5))
print(f"k-NN (k=1), CV: {cv_score_knn1:.3f}")

In [ ]:
rf = RandomForestClassifier(random_state=17, n_jobs=-1)
cv_score_rf = np.mean(cross_val_score(rf, X_train, y_train, cv=5))
print(f"Random Forest, CV: {cv_score_rf:.3f}")

**Итоги по MNIST:**

| Модель | CV | Holdout |
|---|---|---|
| Дерево решений | ~0.844 | ~0.838 |
| k-NN (k=1) | ~0.987 | ~0.983 |
| Случайный лес | ~0.935 | ~0.941 |

На этом датасете k-NN с $k=1$ бьёт случайный лес! Интуиция простая: похожие рукописные цифры действительно выглядят похоже — пиксели совпадают. Евклидово расстояние здесь работает хорошо.

**Вывод:** нет универсально лучшего алгоритма. На одних данных лучше дерево, на других — k-NN. Проверяйте несколько методов.

### 5.4 Сложный случай для k-NN: проклятие размерности

Теперь создадим данные, где k-NN плохо работает. 100 признаков, из которых только один содержит полезную информацию, остальные — шум.

In [ ]:
def form_noisy_data(n_obj=1000, n_feat=100, random_seed=17):
    np.random.seed(random_seed)
    y_noisy = np.random.choice([-1, 1], size=n_obj)
    x1 = 0.3 * y_noisy                                       # один полезный признак
    x_noise = np.random.random(size=(n_obj, n_feat - 1))     # остальное — шум
    return np.hstack([x1.reshape(-1, 1), x_noise]), y_noisy

X_noisy, y_noisy = form_noisy_data()

X_tr, X_ho, y_tr, y_ho = train_test_split(X_noisy, y_noisy, test_size=0.3, random_state=17)

# Смотрим, как accuracy зависит от числа соседей
cv_scores, ho_scores = [], []
k_values = [1, 2, 3, 5] + list(range(50, 550, 50))

for k in k_values:
    pipe = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=k))])
    cv_scores.append(np.mean(cross_val_score(pipe, X_tr, y_tr, cv=5)))
    pipe.fit(X_tr, y_tr)
    ho_scores.append(accuracy_score(y_ho, pipe.predict(X_ho)))

plt.figure(figsize=(9, 4))
plt.plot(k_values, cv_scores, label="CV")
plt.plot(k_values, ho_scores, label="Holdout")
plt.axhline(0.5, color="red", linestyle="--", alpha=0.5, label="Случайное угадывание")
plt.xlabel("n_neighbors (k)"); plt.ylabel("Accuracy")
plt.title("k-NN не справляется: полезный сигнал тонет в шуме")
plt.legend();

In [ ]:
# А вот дерево глубины 1 справляется легко
tree_simple = DecisionTreeClassifier(max_depth=1, random_state=17)
cv_tree = np.mean(cross_val_score(tree_simple, X_tr, y_tr, cv=5))
tree_simple.fit(X_tr, y_tr)
ho_tree = accuracy_score(y_ho, tree_simple.predict(X_ho))
print(f"Дерево (max_depth=1): CV = {cv_tree:.3f}, Holdout = {ho_tree:.3f}")

k-NN почти не лучше случайного угадывания. Почему?

При 100 признаках евклидово расстояние «усредняется» по всем из них. Один полезный признак «тонет» среди 99 шумовых. Все объекты становятся примерно одинаково далеко друг от друга — **проклятие размерности**.

Дерево же на первом же шаге находит единственный полезный признак и строит разбиение по нему.

---
## 6. Плюсы и минусы методов

### Дерево решений

**Плюсы:**
- Интерпретируемость: легко объяснить, почему модель приняла то или иное решение
- Можно визуализировать (как саму модель, так и путь предсказания для конкретного объекта)
- Быстрое обучение и предсказание
- Мало гиперпараметров
- Работает с числовыми и категориальными признаками
- Не требует масштабирования признаков

**Минусы:**
- Чувствительность к шуму: небольшое изменение данных может полностью изменить структуру дерева
- Граница решений — только параллельные осям гиперплоскости → плохо на диагональных/круговых границах
- Склонность к переобучению — нужна регуляризация (`max_depth`, `min_samples_leaf`)
- Нестабильность: дерево может сильно измениться от случайного разбиения данных → решение: ансамбли (Random Forest)
- Не умеет экстраполировать: за пределами обучающего множества предсказывает константу

### Метод ближайших соседей (k-NN)

**Плюсы:**
- Простота реализации и понимания
- Хорошо изучен теоретически: на «бесконечных» данных — оптимальный метод
- Хорошая отправная точка (baseline)
- Легко адаптируется под задачу через выбор метрики
- Работает с мультиклассовой классификацией и регрессией без изменений

**Минусы:**
- Медленные предсказания на больших данных (нужно считать расстояния до всех объектов обучающей выборки)
- Сильно страдает от проклятия размерности: при большом числе признаков расстояния «выравниваются»
- Требует обязательного масштабирования признаков
- Чувствителен к нерелевантным признакам
- Нет явной модели: нельзя «посмотреть» на то, что выучил алгоритм

---
## 7. Полезные ресурсы

- [Документация sklearn: DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
- [Документация sklearn: KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)
- [Визуализация деревьев решений](https://scikit-learn.org/stable/modules/tree.html#tree)
